This script gives a rough idea of the AppML_ZTF_table.csv dataset.

In [ ]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
import pandas as pd 
from pandas import Series, DataFrame 
from matplotlib import pyplot as plt
import numpy as np
import seaborn as sns

SavePlots = False

Import Data

In [ ]:
# Read the data and print the variables:
# data = pd.DataFrame(np.genfromtxt('../data/AppML_ZTF_table.csv', delimiter=',', names=True))
data = pd.read_csv('../data/AppML_ZTF_table.csv')

data.drop(columns=['objectId'], inplace=True)
# variables = data.columns
# print(variables.values)

# also drop some clearly useless variables (e.g. timestamp, host galaxy name, etc.):
data.drop(columns=['decstd', 'rastd', 'classificationReliability', 'host_name',
                   'htm16', 'ssnamenr', 'jdmax', 'jdmin', 'jd_g_minus_r'], inplace=True)
# variables = data.columns
# print(variables.values)


# get classification:
classifications = data[['classification']]
tns_type = data[['tns_type']]
tns_name = data[['tns_name']]
print('Classifications:', np.unique(classifications))
# print('TNS types:', np.unique(tns_type))
# print('TNS names:', np.unique(tns_name))
# drop classification too
data.drop(columns=['classification', 'tns_type', 'tns_name'], inplace=True)
variables = data.columns
print(variables.values)


In [ ]:
# reduced_vars = ['glatmean', 'glonmean', 'rmag', 'dmdt_r']
reduced_vars = ['glatmean', 'glonmean', 'rmag', 'g_minus_r']
reduced_data = data[reduced_vars]
sns.pairplot(reduced_data, vars=reduced_vars, diag_kind='hist', plot_kws={'alpha':0.3})
plt.savefig('../figures/example_pairplot.png', dpi=300)

Inspect data in 1D histograms, to determine interesting variables

In [ ]:
nvars = len(variables)
print(f'Number of variables: {nvars}')

fig, axes = plt.subplots(int(np.ceil(nvars/5)), 5, figsize=(15, 18))
# def get_color(col):

for i, axis in enumerate(axes.reshape(-1)):
    if i >= nvars:
        axis.axis('off')
        continue
    param = variables.values[i]

    # check if a logarithmic scale is needed:
    std = data[param].std()
    mean = data[param].mean()
    max_val = data[param].max()
    min_val = data[param].min()

    axis.text(0.5, 0.9, param, transform=axis.transAxes, ha='center', va='center')
    axis.set_yticks([])

    if param == 'dmdt_g':
        print(f'{param}: mean={mean}, std={std}, max={max_val}, min={min_val}')
    try:
        if (abs(max_val-mean)/std > 10) or (abs(mean-min_val)/std > 10):
            # if centered around zero, use symlog, otherwise log
            if abs(mean) < std:
                axis.set_xscale('symlog')
            else:    
                axis.set_xscale('log')

            log_data = np.log10(data[param].values)
            axis.hist(log_data, bins=50, color='cornflowerblue', alpha=1)

        else:
            axis.hist(data[param].values, bins=50, color='cornflowerblue', alpha=1)
    except:
        # plot ERRORS in red:
        axis.text(0.5, 0.5, 'ERROR', transform=axis.transAxes, ha='center', va='center', color='red')

fig.tight_layout()
plt.savefig('../figures/variable_distributions.png', dpi=300)
# plt.show()
plt.clf()